In [ ]:
MERGE INTO openalex.works.work_topics_frontfill AS target
USING (
  WITH topics_metadata AS (
    SELECT
      topic_id, t.display_name,
      NAMED_STRUCT('id', concat('https://openalex.org/subfields/', s.subfield_id), 'display_name', s.display_name) AS subfield,
      NAMED_STRUCT('id', concat('https://openalex.org/fields/', f.field_id), 'display_name', f.display_name) AS field,
      NAMED_STRUCT('id', concat('https://openalex.org/domains/', d.domain_id), 'display_name', d.display_name) AS domain
    FROM openalex.common.topics t
    JOIN openalex.common.subfields s USING (subfield_id)
    JOIN openalex.common.fields f USING (field_id)
    JOIN openalex.common.domains d USING (domain_id)
  )
  SELECT
    work_id,
    array_sort(array_agg(
      NAMED_STRUCT(
        'id', concat('https://openalex.org/T', topic_id),
        'display_name', tm.display_name,
        'score', ROUND(score, 4),
        'subfield', tm.subfield,
        'field', tm.field,
        'domain', tm.domain
      )
    ), (left, right) -> CASE
      WHEN left.score > right.score THEN -1
      WHEN left.score < right.score THEN 1
      WHEN left.id < right.id THEN -1
      WHEN left.id > right.id THEN 1
      ELSE 0
    END) as topics,
    first(wt.source) as source,
    max(wt.created_datetime) as created_datetime,
    max(wt.updated_datetime) as updated_datetime
  FROM openalex.works.work_topic wt
  JOIN topics_metadata tm USING (topic_id)
  GROUP BY work_id
) AS source
ON target.work_id = source.work_id
WHEN NOT MATCHED THEN INSERT (work_id, topics, source, created_datetime, updated_datetime)
VALUES (source.work_id, source.topics, source.source, source.created_datetime, source.updated_datetime);

# Topics: Merge Backfill (One-Time Migration)

Merges legacy topic assignments from `work_topic` (original MAG/OpenAlex data) into `work_topics_frontfill`. Enriches raw topic IDs with hierarchy metadata.

Uses `WHEN NOT MATCHED` only — never overwrites existing BERT predictions. This is intended as a one-time migration to seed `work_topics_frontfill` with ~239.5M legacy rows. Not part of the scheduled pipeline.

In [ ]:
OPTIMIZE openalex.works.work_topics_frontfill;